# Conformance checking per tipo di oggetto

Questo notebook verifica la conformità delle proiezioni degli
oggetti strutturali rispetto alle Petri net componenti della
Object-Centric Petri Net scoperta da PM4Py.

Il controllo utilizza il token-based replay sulle prospettive
`orders`, `items` e `packages`.

## Obiettivo e perimetro

PM4Py rappresenta la OCPN scoperta come una collezione di Petri
net tradizionali, una per ogni tipo di oggetto.

L'OCEL viene quindi appiattito separatamente per ciascun tipo.
Ogni oggetto diventa un caso e il relativo ciclo di vita viene
confrontato con la Petri net corrispondente.

Il risultato costituisce un controllo formale delle singole
proiezioni. Non rappresenta una fitness object-centric globale
sincronizzata tra tipi di oggetto differenti.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import pm4py


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

source_directory = project_root / "src"

if str(source_directory) not in sys.path:
    sys.path.insert(
        0,
        str(source_directory),
    )


from ocpm_partial_order.config import (
    MAIN_DATASET_DB,
)
from ocpm_partial_order.conformance import (
    check_execution_projections,
    check_object_projection,
    check_object_type_conformance,
    replay_flattened_object_trace,
)
from ocpm_partial_order.discovery import (
    discover_ocpn,
)
from ocpm_partial_order.io.ocel_loader import (
    load_ocel2_sqlite,
)


print("PM4Py version:", pm4py.__version__)
print("Dataset:", MAIN_DATASET_DB)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




PM4Py version: 2.7.23.3
Dataset: C:\Users\nicol\ocpm-partial-order\ocpm-partial-order\data\raw\order_management.sqlite


## Caricamento del log e scoperta del modello

La OCPN viene scoperta dall'intero dataset Order Management.
Le Petri net relative a ordini, articoli e pacchi verranno
utilizzate per il token-based replay.

In [2]:
ocel = load_ocel2_sqlite(
    MAIN_DATASET_DB
)

ocpn = discover_ocpn(
    ocel
)

available_types = sorted(
    ocpn["petri_nets"]
)

print(
    "Object types disponibili:",
    available_types,
)

Object types disponibili: ['employees', 'items', 'orders', 'packages', 'products']


## Process execution di riferimento

Il primo controllo riguarda l'ordine `o-990424`, già utilizzato
per verificare la costruzione dell'Instance Graph.

La process execution comprende un ordine, un articolo e un
pacco. Ciascun oggetto viene controllato rispetto alla propria
Petri net componente.

In [3]:
reference_objects = {
    "orders": {
        "o-990424",
    },
    "items": {
        "i-881734",
    },
    "packages": {
        "p-660247",
    },
}


execution_report = (
    check_execution_projections(
        ocel=ocel,
        ocpn=ocpn,
        object_ids_by_type=(
            reference_objects
        ),
    )
)


execution_rows = [
    {
        "object_type": (
            result.object_type
        ),
        "object_id": (
            result.object_id
        ),
        "events": (
            result.event_count
        ),
        "activities": " -> ".join(
            result.activities
        ),
        "trace_is_fit": (
            result.trace_is_fit
        ),
        "trace_fitness": (
            result.trace_fitness
        ),
        "missing_tokens": (
            result.missing_tokens
        ),
        "remaining_tokens": (
            result.remaining_tokens
        ),
    }
    for result
    in execution_report.trace_results
]


execution_table = pd.DataFrame(
    execution_rows
)

execution_table

,object_type,object_id,events,activities,trace_is_fit,trace_fitness,missing_tokens,remaining_tokens
0,items,i-881734,7,place order -> confirm order -> pick item -> c...,True,1.0,0,0
1,orders,o-990424,3,place order -> confirm order -> pay order,True,1.0,0,0
2,packages,p-660247,3,create package -> send package -> package deli...,True,1.0,0,0


In [4]:
print(
    "All projections fit:",
    execution_report.all_projections_fit,
)
print(
    "Minimum projection fitness:",
    execution_report.minimum_trace_fitness,
)
print(
    "Average projection fitness:",
    execution_report.average_trace_fitness,
)

All projections fit: True
Minimum projection fitness: 1.0
Average projection fitness: 1.0


## Controllo dell'intero dataset

Il controllo viene ora esteso a tutti gli oggetti dei tre tipi
strutturali.

Le tracce appartengono allo stesso OCEL utilizzato per la
discovery della OCPN. Si tratta quindi di una valutazione
in-sample.

In [5]:
structural_object_types = (
    "orders",
    "items",
    "packages",
)


summaries = [
    check_object_type_conformance(
        ocel=ocel,
        ocpn=ocpn,
        object_type=object_type,
    )
    for object_type
    in structural_object_types
]


summary_rows = [
    {
        "object_type": (
            summary.object_type
        ),
        "traces": (
            summary.trace_count
        ),
        "fitting_traces": (
            summary.fitting_trace_count
        ),
        "non_fitting_traces": (
            summary.non_fitting_trace_count
        ),
        "fitting_percentage": (
            summary.fitting_percentage
        ),
        "average_trace_fitness": (
            summary.average_trace_fitness
        ),
        "missing_tokens": (
            summary.total_missing_tokens
        ),
        "remaining_tokens": (
            summary.total_remaining_tokens
        ),
    }
    for summary in summaries
]


summary_table = pd.DataFrame(
    summary_rows
)

summary_table

,object_type,traces,fitting_traces,non_fitting_traces,fitting_percentage,average_trace_fitness,missing_tokens,remaining_tokens
0,orders,2000,2000,0,100.0,1.0,0,0
1,items,7659,7659,0,100.0,1.0,0,0
2,packages,1128,1128,0,100.0,1.0,0,0


In [6]:
total_projections = sum(
    summary.trace_count
    for summary in summaries
)

total_fitting = sum(
    summary.fitting_trace_count
    for summary in summaries
)

total_non_fitting = sum(
    summary.non_fitting_trace_count
    for summary in summaries
)

total_missing = sum(
    summary.total_missing_tokens
    for summary in summaries
)

total_remaining = sum(
    summary.total_remaining_tokens
    for summary in summaries
)


overall_summary = pd.DataFrame(
    [
        {
            "structural_projections": (
                total_projections
            ),
            "fitting_projections": (
                total_fitting
            ),
            "non_fitting_projections": (
                total_non_fitting
            ),
            "missing_tokens": (
                total_missing
            ),
            "remaining_tokens": (
                total_remaining
            ),
        }
    ]
)

overall_summary

,structural_projections,fitting_projections,non_fitting_projections,missing_tokens,remaining_tokens
0,10787,10787,0,0,0


## Controllo negativo

Per verificare che il token-based replay distingua una traccia
conforme da una non conforme, viene rimossa l'attività finale
`package delivered` dal ciclo di vita del pacco `p-660247`.

La traccia originale deve avere fitness pari a 1. La versione
incompleta deve produrre token mancanti e residui.

In [7]:
flattened_packages = (
    pm4py.ocel_flattening(
        ocel,
        "packages",
    )
)


original_package_trace = (
    flattened_packages[
        flattened_packages[
            "case:concept:name"
        ].astype(str)
        == "p-660247"
    ].copy()
)


original_result = (
    check_object_projection(
        ocel=ocel,
        ocpn=ocpn,
        object_type="packages",
        object_id="p-660247",
    )
)


incomplete_package_trace = (
    original_package_trace.iloc[:-1]
    .copy()
)


incomplete_result = (
    replay_flattened_object_trace(
        flattened_trace=(
            incomplete_package_trace
        ),
        component_petri_net=(
            ocpn[
                "petri_nets"
            ]["packages"]
        ),
        object_type="packages",
        object_id="p-660247",
    )
)


negative_control_table = pd.DataFrame(
    [
        {
            "scenario": "original",
            "activities": " -> ".join(
                original_result.activities
            ),
            "trace_is_fit": (
                original_result.trace_is_fit
            ),
            "trace_fitness": (
                original_result.trace_fitness
            ),
            "missing_tokens": (
                original_result.missing_tokens
            ),
            "remaining_tokens": (
                original_result.remaining_tokens
            ),
        },
        {
            "scenario": "missing final event",
            "activities": " -> ".join(
                incomplete_result.activities
            ),
            "trace_is_fit": (
                incomplete_result.trace_is_fit
            ),
            "trace_fitness": (
                incomplete_result.trace_fitness
            ),
            "missing_tokens": (
                incomplete_result.missing_tokens
            ),
            "remaining_tokens": (
                incomplete_result.remaining_tokens
            ),
        },
    ]
)

negative_control_table

,scenario,activities,trace_is_fit,trace_fitness,missing_tokens,remaining_tokens
0,original,create package -> send package -> package deli...,True,1.000000,0,0
1,missing final event,create package -> send package,False,0.666667,1,1


## Interpretazione

Tutte le proiezioni osservate di ordini, articoli e pacchi
risultano conformi alle rispettive Petri net componenti.

Il controllo negativo dimostra che il risultato non deriva da
un meccanismo che assegna sempre fitness pari a 1. Quando
l'evento finale viene rimosso, la traccia non raggiunge
correttamente il final marking e il token-based replay segnala
la deviazione.

In [8]:
assert (
    execution_report
    .all_projections_fit
)

assert (
    execution_report
    .minimum_trace_fitness
    == 1.0
)

assert total_projections == 10787
assert total_fitting == 10787
assert total_non_fitting == 0
assert total_missing == 0
assert total_remaining == 0

assert original_result.trace_is_fit
assert (
    original_result.trace_fitness
    == 1.0
)

assert not (
    incomplete_result.trace_is_fit
)
assert (
    incomplete_result.trace_fitness
    < 1.0
)
assert (
    incomplete_result.missing_tokens
    > 0
)
assert (
    incomplete_result.remaining_tokens
    > 0
)

print(
    "Conformance notebook validation: "
    "PASSED"
)

Conformance notebook validation: PASSED


## Limiti e conclusioni

Il token-based replay dimostra la conformità delle proiezioni
per tipo di oggetto. Non verifica però una semantica globale
nella quale ordini, articoli e pacchi vengono sincronizzati
contemporaneamente.

Inoltre, la OCPN è stata scoperta dallo stesso OCEL utilizzato
per il controllo. I risultati misurano quindi la fitness
in-sample delle Petri net componenti.

L'esperimento fornisce comunque una verifica formale e
riproducibile che completa i precedenti controlli strutturali
sugli Instance Graph.